# Modelado de Machine Learning
## Prediccion de Precio de Alquiler — Ecuador
---
**Laboratorio de Ciencia de Datos — Proceso de Seleccion Tecnico de Investigacion**
**Escuela Politecnica Nacional — Laboratorio ADA**

Este notebook cubre el **Paso 2**:
- Ingenieria de features
- Pipeline de preprocesamiento con `sklearn`
- Comparacion de seis modelos de regresion
- Evaluacion con metricas adecuadas (MAE, RMSE, R², MAPE)
- Serializacion del modelo final con `joblib`
- Generacion de los archivos de la API REST (FastAPI)


In [ ]:
!pip install scikit-learn xgboost lightgbm joblib shap --quiet

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib, shap

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")
os.makedirs("figs",  exist_ok=True)
os.makedirs("model", exist_ok=True)

COLOR_MAIN = "#1a5276"
COLOR_ACC  = "#2e86c1"
COLOR_WARN = "#e74c3c"
print("Librerias cargadas")


## 1. Carga del Dataset Limpio

In [ ]:
CLEAN_PATH = "data/alquileres_clean.csv"
df = pd.read_csv(CLEAN_PATH, encoding="utf-8")

FEATURES = ["provincia","lugar","num_dormitorios","num_banos","area","num_garages"]
TARGET   = "precio"

df = df[FEATURES + [TARGET]].dropna(subset=[TARGET])
print(f"Dataset: {df.shape[0]:,} registros — {len(FEATURES)} features")
df.describe().round(2)


## 2. Ingenieria de Features

In [ ]:
# Features derivadas con justificacion semantica:
# - banos_por_dorm: densidad de banos relativa al tamano
# - area_por_dorm:  area util por dormitorio
# - tiene_garaje:   indicador binario de amenidad premium
# - log_area:       transformacion para linealizar la relacion area-precio

df["banos_por_dorm"] = df["num_banos"]    / (df["num_dormitorios"] + 1)
df["area_por_dorm"]  = df["area"]         / (df["num_dormitorios"] + 1)
df["tiene_garaje"]   = (df["num_garages"] > 0).astype(int)
df["log_area"]       = np.log1p(df["area"])

FEATURES_ENG = FEATURES + ["banos_por_dorm","area_por_dorm","tiene_garaje","log_area"]
CAT_FEATURES = ["provincia","lugar"]
NUM_FEATURES = [f for f in FEATURES_ENG if f not in CAT_FEATURES]

print("Features categoricas:", CAT_FEATURES)
print("Features numericas  :", NUM_FEATURES)


## 3. Pipeline de Preprocesamiento

In [ ]:
# Pipeline con ColumnTransformer:
# - Numericas: imputacion por mediana + estandarizacion
# - Categoricas: imputacion por moda + OrdinalEncoder robusto
#
# Transformacion del target: log(1+precio) para simetria y
# reduccion del impacto de outliers en la funcion de perdida.

num_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

cat_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer([
    ("num", num_transformer, NUM_FEATURES),
    ("cat", cat_transformer, CAT_FEATURES),
], remainder="drop")

X     = df[FEATURES_ENG].copy()
y_raw = df[TARGET].values
y     = np.log1p(y_raw)   # log-transform del target

print("Shape X:", X.shape)
print("Shape y:", y.shape)
print(f"Precio: rango [{y_raw.min():.0f}, {y_raw.max():.0f}] USD")
print(f"log(precio): rango [{y.min():.3f}, {y.max():.3f}]")


## 4. Division Train / Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train : {X_train.shape[0]:,}")
print(f"Test  : {X_test.shape[0]:,}")


## 5. Comparacion de Modelos

### Justificacion de la seleccion de candidatos

| Familia | Modelos | Razon de inclusion |
|---|---|---|
| Regresion lineal regularizada | Ridge, Lasso | Baseline interpretable; penaliza colinealidad |
| Arboles de decision ensamblados | RandomForest, ExtraTrees | Robustez ante outliers, captura interacciones no lineales |
| Gradient Boosting moderno | XGBoost, LightGBM | Estado del arte en datos tabulares; LGBM maneja alta cardinalidad categorica eficientemente |

La seleccion final se basa en RMSE y R² en validacion cruzada 5-Fold.
El target en escala log permite comparar modelos lineales y no-lineales en igualdad de condiciones.


In [ ]:
candidatos = {
    "Ridge"       : Pipeline([("pre", preprocessor), ("mdl", Ridge(alpha=10.0))]),
    "Lasso"       : Pipeline([("pre", preprocessor), ("mdl", Lasso(alpha=0.01))]),
    "RandomForest": Pipeline([("pre", preprocessor), ("mdl", RandomForestRegressor(
                                 n_estimators=300, min_samples_leaf=3,
                                 random_state=42, n_jobs=-1))]),
    "ExtraTrees"  : Pipeline([("pre", preprocessor), ("mdl", ExtraTreesRegressor(
                                 n_estimators=300, random_state=42, n_jobs=-1))]),
    "XGBoost"     : Pipeline([("pre", preprocessor), ("mdl", XGBRegressor(
                                 n_estimators=500, learning_rate=0.05, max_depth=6,
                                 subsample=0.8, colsample_bytree=0.8,
                                 random_state=42, n_jobs=-1, verbosity=0))]),
    "LightGBM"    : Pipeline([("pre", preprocessor), ("mdl", LGBMRegressor(
                                 n_estimators=500, learning_rate=0.05, num_leaves=63,
                                 random_state=42, n_jobs=-1, verbose=-1))]),
}

kf         = KFold(n_splits=5, shuffle=True, random_state=42)
resultados = []

print(f"{'Modelo':<14}  {'CV RMSE (log)':>15}  {'CV R2':>10}")
print("-" * 45)

for nombre, pipe in candidatos.items():
    cv_r2   = cross_val_score(pipe, X_train, y_train, cv=kf,
                               scoring="r2", n_jobs=-1)
    cv_rmse = cross_val_score(pipe, X_train, y_train, cv=kf,
                               scoring="neg_root_mean_squared_error", n_jobs=-1)
    resultados.append({
        "Modelo"     : nombre,
        "CV_RMSE"    : -cv_rmse.mean(),
        "CV_RMSE_std": cv_rmse.std(),
        "CV_R2"      : cv_r2.mean(),
        "CV_R2_std"  : cv_r2.std(),
    })
    print(f"{nombre:<14}  {-cv_rmse.mean():>8.4f} +/-{cv_rmse.std():.4f}  "
          f"  R2={cv_r2.mean():.4f} +/-{cv_r2.std():.4f}")

df_res = pd.DataFrame(resultados).sort_values("CV_RMSE")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = sns.color_palette("Blues_r", len(df_res))

df_r = df_res.sort_values("CV_RMSE")
axes[0].barh(df_r["Modelo"], df_r["CV_RMSE"],
             xerr=df_r["CV_RMSE_std"], color=colors, edgecolor="white")
axes[0].set_xlabel("CV RMSE (escala log)")
axes[0].set_title("Comparacion por RMSE (menor = mejor)")
axes[0].axvline(df_r["CV_RMSE"].min(), color=COLOR_WARN, lw=1.5, ls="--")

df_r2 = df_res.sort_values("CV_R2")
axes[1].barh(df_r2["Modelo"], df_r2["CV_R2"],
             xerr=df_r2["CV_R2_std"], color=colors, edgecolor="white")
axes[1].set_xlabel("CV R2")
axes[1].set_title("Comparacion por R2 (mayor = mejor)")
axes[1].axvline(df_r2["CV_R2"].max(), color=COLOR_WARN, lw=1.5, ls="--")

plt.suptitle("Evaluacion 5-Fold CV — Candidatos de Regresion",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figs/comparacion_modelos.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nMejor modelo:")
print(df_res.iloc[0])


## 6. Entrenamiento del Modelo Final

In [ ]:
mejor_nombre  = df_res.iloc[0]["Modelo"]
print(f"Modelo seleccionado: {mejor_nombre}")

pipeline_final = candidatos[mejor_nombre]
pipeline_final.fit(X_train, y_train)

# Evaluacion sobre test set (escala original USD)
y_pred_log = pipeline_final.predict(X_test)
y_pred_usd = np.expm1(y_pred_log)
y_test_usd = np.expm1(y_test)

mae  = mean_absolute_error(y_test_usd, y_pred_usd)
rmse = np.sqrt(mean_squared_error(y_test_usd, y_pred_usd))
r2   = r2_score(y_test_usd, y_pred_usd)
mape = np.mean(np.abs((y_test_usd - y_pred_usd) / y_test_usd)) * 100

print("=" * 50)
print("METRICAS SOBRE TEST SET (escala USD)")
print("=" * 50)
print(f"  MAE   (Error Absoluto Medio)        : ${mae:,.2f}")
print(f"  RMSE  (Raiz Error Cuadratico Medio) : ${rmse:,.2f}")
print(f"  MAPE  (Error Porcentual Absoluto)   : {mape:.2f}%")
print(f"  R2    (Coeficiente de determinacion): {r2:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Predicho vs Real
axes[0].scatter(y_test_usd, y_pred_usd, alpha=0.4, s=12, color=COLOR_ACC)
lims = [min(y_test_usd.min(), y_pred_usd.min()),
        max(y_test_usd.max(), y_pred_usd.max())]
axes[0].plot(lims, lims, "r--", lw=1.5, label="Prediccion perfecta")
axes[0].set_xlabel("Precio real (USD)")
axes[0].set_ylabel("Precio predicho (USD)")
axes[0].set_title(f"Predicho vs Real\n(R2 = {r2:.4f})")
axes[0].legend()

# Residuos
residuos = y_test_usd - y_pred_usd
axes[1].scatter(y_pred_usd, residuos, alpha=0.4, s=12, color=COLOR_MAIN)
axes[1].axhline(0, color="red", lw=1.5, ls="--")
axes[1].set_xlabel("Precio predicho (USD)")
axes[1].set_ylabel("Residuo (USD)")
axes[1].set_title("Residuos vs Prediccion")

# Distribucion de residuos
axes[2].hist(residuos, bins=50, color=COLOR_ACC, edgecolor="white")
axes[2].axvline(0, color="red", lw=1.5, ls="--")
axes[2].set_xlabel("Residuo (USD)")
axes[2].set_ylabel("Frecuencia")
axes[2].set_title(f"Distribucion de residuos\n(MAE = ${mae:,.0f})")

plt.suptitle(f"Evaluacion del modelo: {mejor_nombre}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figs/evaluacion_modelo.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Importancia de Variables (SHAP)

In [ ]:
print("Calculando valores SHAP...")

X_test_pre      = pipeline_final.named_steps["pre"].transform(X_test)
model_only      = pipeline_final.named_steps["mdl"]
feature_names_pre = NUM_FEATURES + CAT_FEATURES

try:
    explainer = shap.TreeExplainer(model_only)
    shap_vals = explainer.shap_values(X_test_pre)

    fig, ax = plt.subplots(figsize=(9, 5))
    shap.summary_plot(shap_vals, X_test_pre,
                      feature_names=feature_names_pre,
                      max_display=12, show=False, plot_type="bar")
    plt.title(f"Importancia de variables (SHAP) — {mejor_nombre}", fontsize=12)
    plt.tight_layout()
    plt.savefig("figs/shap_importancia.png", dpi=150, bbox_inches="tight")
    plt.show()
except Exception as e:
    print(f"TreeExplainer no disponible: {e}. Usando feature_importances_")
    if hasattr(model_only, "feature_importances_"):
        imp = pd.Series(model_only.feature_importances_, index=feature_names_pre)
        imp.sort_values().plot(kind="barh", color=COLOR_ACC, figsize=(9, 5))
        plt.title(f"Importancia de variables — {mejor_nombre}")
        plt.tight_layout()
        plt.savefig("figs/feature_importance.png", dpi=150, bbox_inches="tight")
        plt.show()


## 8. Serializacion del Modelo

In [ ]:
# Reentrenar sobre el 100% de datos antes de serializar
print("Reentrenando sobre dataset completo...")
pipeline_final.fit(X, y)
print("Listo")

modelo_info = {
    "pipeline"    : pipeline_final,
    "features"    : FEATURES_ENG,
    "target"      : TARGET,
    "log_transform": True,
    "modelo_nombre": mejor_nombre,
    "metricas_test": {"MAE":mae,"RMSE":rmse,"R2":r2,"MAPE":mape},
    "categorias"  : {
        "provincia": df["provincia"].unique().tolist(),
        "lugar"    : df["lugar"].unique().tolist(),
    }
}

MODEL_PATH = "model/modelo_alquileres.joblib"
joblib.dump(modelo_info, MODEL_PATH, compress=3)
print(f"Modelo serializado en: {MODEL_PATH}")
print(f"Tamano: {os.path.getsize(MODEL_PATH)/1024:.1f} KB")


In [ ]:
# Verificacion de carga y prediccion de ejemplo
loaded = joblib.load(MODEL_PATH)
pipe   = loaded["pipeline"]

ejemplo = pd.DataFrame([{
    "provincia"      : "Pichincha",
    "lugar"          : "Quito",
    "num_dormitorios": 3,
    "num_banos"      : 2,
    "area"           : 120,
    "num_garages"    : 1,
    "banos_por_dorm" : 2 / (3 + 1),
    "area_por_dorm"  : 120 / (3 + 1),
    "tiene_garaje"   : 1,
    "log_area"       : np.log1p(120),
}])

pred_usd = np.expm1(pipe.predict(ejemplo)[0])
print(f"Prediccion de ejemplo: ${pred_usd:,.2f} USD/mes")
print(f"\nModelo cargado: {loaded['modelo_nombre']}")
print(f"Metricas (test set): {loaded['metricas_test']}")


## 9. Estructura de la API REST (FastAPI)

Los archivos de la API se generan a continuacion.
El modelo serializado (`model/modelo_alquileres.joblib`) se referencia directamente.

### Endpoint principal

```
POST /predict
Content-Type: application/json

{
  "provincia": "Pichincha",
  "lugar": "Quito",
  "num_dormitorios": 3,
  "num_banos": 2,
  "area": 120,
  "num_garages": 1
}
```

Respuesta:
```json
{"prediction": 750.0}
```


In [ ]:
os.makedirs("api", exist_ok=True)

requirements = (
    "fastapi>=0.110.0\n"
    "uvicorn[standard]>=0.29.0\n"
    "scikit-learn>=1.4.0\n"
    "xgboost>=2.0.0\n"
    "lightgbm>=4.3.0\n"
    "joblib>=1.3.0\n"
    "numpy>=1.26.0\n"
    "pandas>=2.2.0\n"
    "python-multipart>=0.0.9\n"
)
with open("api/requirements.txt", "w") as f:
    f.write(requirements)
print("api/requirements.txt creado")

# Copiar main.py desde el repositorio (archivo pre-generado)
import shutil, urllib.request

# Si se ejecuta en Colab con el repo clonado:
# shutil.copy("api/main.py", "api/main.py")

print("Estructura de api/:")
print("  api/")
print("    main.py          <- FastAPI application")
print("    requirements.txt <- dependencias")
print("  model/")
print("    modelo_alquileres.joblib")
print("  Dockerfile")
print("  render.yaml")
print("  README.md")


In [ ]:
# Test local (requiere uvicorn instalado)
# Para ejecutar: uvicorn api.main:app --reload
# Documentacion interactiva: http://localhost:8000/docs

# Ejemplo con requests (una vez desplegado):
# import requests
# r = requests.post(
#     "https://<tu-servicio>.onrender.com/predict",
#     json={
#         "provincia": "Pichincha", "lugar": "Quito",
#         "num_dormitorios": 3, "num_banos": 2,
#         "area": 120, "num_garages": 1
#     }
# )
# print(r.json())  # {"prediction": 750.0}

print("Consultar README.md para instrucciones completas de despliegue.")


---
## Resumen del Modelado

| Elemento | Descripcion |
|---|---|
| Candidatos evaluados | Ridge, Lasso, RandomForest, ExtraTrees, XGBoost, LightGBM |
| Seleccion | Menor RMSE en 5-Fold CV |
| Target | log(1+precio) — recuperado con expm1() |
| Features derivadas | banos_por_dorm, area_por_dorm, tiene_garaje, log_area |
| Encoder categorico | OrdinalEncoder robusto (unknown_value=-1) |
| Metricas de evaluacion | MAE, RMSE, R2, MAPE (en escala USD) |
| Archivo serializado | model/modelo_alquileres.joblib |

**Siguiente paso:** despliegue de la API en Render / Railway / Fly.io siguiendo `README.md`
